## Cell 1: Importing Libraries

This cell imports all the necessary Python libraries required for the project:

| Library | Purpose |
|---|---|
| `numpy` | Numerical computing, array operations |
| `pandas` | Data loading, manipulation, and analysis |
| `requests` | HTTP requests (imported but not used) |
| `pickle` | Saving/loading the trained model to disk |
| `re` | Regular expressions for text cleaning |
| `scipy.sparse` | Handling sparse matrices (imported but not used directly) |
| `train_test_split` | Splitting dataset into train and test sets |
| `LogisticRegression` | The ML classification algorithm |
| `TfidfVectorizer` | Converts text to numerical TF-IDF features |
| `PorterStemmer` | Reduces words to their root/stem form |
| `stopwords` | List of common English words to ignore |
| `accuracy_score` | Evaluates model performance |
| `nltk.download` | Downloads required NLTK language data files |

In [1]:
import numpy as np
import pandas as pd
import requests
import pickle
import re
import scipy.sparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from sklearn.metrics import accuracy_score
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## Cell 2: Viewing English Stopwords

This cell prints all English stopwords from the NLTK library.

**Stopwords** are common words like *"a", "the", "is", "and"* that carry no meaningful sentiment information. Removing them reduces noise and improves model performance.

> **Example:** The sentence *"I am not happy"* after stopword removal becomes *"happy"* — only the meaningful word remains.

In [2]:
print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

DATA Processing

## Cell 3: Loading the Dataset

This cell loads the Twitter Sentiment140 dataset from a CSV file.

- **`column_names`**: Manually assigned since the CSV has no header row.
- **`encoding='ISO-8859-1'`**: Required because the raw data contains non-UTF-8 characters (special characters in tweets).

The 6 columns are:

| Column | Description |
|---|---|
| `target` | Sentiment label: **0** = Negative, **4** = Positive |
| `id` | Unique tweet ID |
| `date` | Timestamp of the tweet |
| `flag` | Query used (always `NO_QUERY`) |
| `user` | Twitter username |
| `text` | The actual tweet content |

In [3]:
column_names = ['target', 'id', 'date', 'flag', 'user', 'text']
twitter_data = pd.read_csv(
    'twitter_data.csv',
    names=column_names,
    encoding='ISO-8859-1'
)

## Cell 4: Dataset Shape

Checks the **dimensions** of the loaded dataset.

- Output: `(1600000, 6)` — **1.6 million tweets** with **6 columns**.
- This is a large-scale dataset suitable for training a robust sentiment classifier.

In [4]:
# checking the number of rows and columns
twitter_data.shape

(1600000, 6)

## Cell 5: Preview the First 5 Rows

Displays the **first 5 rows** of the dataset using `.head()`.

This helps verify:
- Data loaded correctly
- Column names are properly assigned
- Tweet text looks as expected

In [5]:
#print 5 rows of dataset
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


## Cell 6: Checking for Missing Values

Uses `.isnull().sum()` to count missing (NaN) values in each column.

- **Output shows all zeros** — no missing values detected.
- This confirms the dataset is complete and ready for preprocessing.

In [7]:
# counting the number of missing valuse in the dataset
twitter_data.isnull().sum()

target    0
id        0
date      0
flag      0
user      0
text      0
dtype: int64

## Cell 7: Target Column Distribution

Checks how many tweets belong to each sentiment class.

- **Output:** 800,000 Negative (`0`) and 800,000 Positive (`4`)
- The dataset is **perfectly balanced** — equal samples for both classes.
- A balanced dataset prevents the model from being biased toward one class.

In [8]:
# checking the distribution of target column
twitter_data['target'].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

convert target 4 to 1


## Cell 8: Converting Target Labels

Replaces the positive label from **4** to **1** using `.replace()`.

**Why?**
- Standard binary classification uses labels **0** and **1**.
- Label `4` is non-standard and could cause confusion.
- After this step: `0 = Negative`, `1 = Positive`

In [9]:
twitter_data.replace({'target': {4: 1}}, inplace=True)

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
1599995,1,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,1,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,1,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,1,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...


## Cell 9: Verify Label Conversion

Verifies that the label replacement worked correctly.

- **Output:** 800,000 rows with label `0` and 800,000 with label `1`
- Confirms the dataset now uses standard binary labels.

In [10]:
# Checking the distribution of target column
twitter_data['target'].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

**Stemming**

Stemming is the process of reducing a word to ist Root word

## Cell 10: Initialize Porter Stemmer

Creates an instance of **PorterStemmer** from NLTK.

The Porter Stemmer reduces words to their base/root form:
- "running" → "run"
- "happily" → "happi"
- "studies" → "studi"

This reduces vocabulary size and groups similar words together, improving model generalization.

In [16]:
post_stem=PorterStemmer()

## Cell 11: Defining the Stemming Function

Defines a custom `stemming()` function that preprocesses each tweet through these steps:

1. **NaN check** — Returns empty string `''` if content is a float (NaN value)
2. **Remove non-alphabetic characters** — `re.sub('[^a-zA-Z]', ' ', content)` strips numbers, punctuation, URLs, @mentions, hashtags
3. **Lowercase + split** — Converts to lowercase and splits into individual words
4. **Stopword removal + stemming** — Removes common English stopwords, then applies Porter stemming to each remaining word
5. **Rejoin** — Returns the cleaned words as a single string joined by spaces

> **Example:** `"@John I was running happily!"` → `"john run happili"`

In [ ]:
stop_words = set(stopwords.words('english'))

def stemming(content):
    if isinstance(content, float):
        return ''
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower().split()
    stemmed_content = [post_stem.stem(word) for word in stemmed_content if word not in stop_words]
    return ' '.join(stemmed_content)9

## Cell 12: Apply Stemming to All Tweets

Applies the `stemming()` function to every tweet in the `text` column and stores results in a new column called **`stemmed_content`**.

- Uses `.apply()` to run the function row-by-row.
- Creates the cleaned, preprocessed text used for model training.
- Processing 1.6 million tweets takes several minutes.

In [18]:
twitter_data['stemmed_content']=twitter_data['text'].apply(stemming)

## Cell 13: Preview Stemmed Data

Displays the first 5 rows again — now including the new **`stemmed_content`** column.

Comparison of original vs stemmed text:
- **Original:** `"is upset that he can't update his Facebook by texting..."`
- **Stemmed:** `"upset updat facebook text might cri result"` — stopwords removed, words reduced to roots

In [19]:
twitter_data.head()

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


## Cell 14: Print All Stemmed Content

Prints the full `stemmed_content` column to verify stemming was applied across all 1.6 million rows.

Shows samples from the beginning (negative tweets) and end (positive tweets) of the dataset.

In [20]:
print(twitter_data['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: str


## Cell 15: Print Target Labels

Prints the `target` column to verify labels are correctly assigned:
- First rows: `0` (Negative)
- Last rows: `1` (Positive)

Confirms data integrity before splitting into features and labels.

In [21]:
print(twitter_data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


## Cell 16: Separating Features and Labels

Splits the data into:
- **`X`** — Input features: the stemmed tweet text as a NumPy array
- **`Y`** — Output labels: `0` or `1` as a NumPy array

This separation is required for supervised machine learning — the model learns the mapping from X (text) → Y (sentiment).

In [22]:
# Seprating the data and label
X=twitter_data['stemmed_content'].values
Y=twitter_data['target'].values

## Cell 17: Preview Feature Array (X)

Prints the `X` array containing all preprocessed tweet texts.

- Each element is a clean, stemmed string ready for vectorization.
- Shape: `(1600000,)` — 1.6 million text samples.

In [23]:
print(X)

<StringArray>
[   'switchfoot http twitpic com zl awww bummer shoulda got david carr third day',
              'upset updat facebook text might cri result school today also blah',
                          'kenichan dive mani time ball manag save rest go bound',
                                                'whole bodi feel itchi like fire',
                                                  'nationwideclass behav mad see',
                                                            'kwesidei whole crew',
                                                                       'need hug',
                      'loltrish hey long time see ye rain bit bit lol fine thank',
                                                                 'tatiana k nope',
                                                             'twittera que muera',
 ...
                                                               'wooooo xbox back',
 'rmedina latati mmmm sound absolut perfect schedul full time lay be

## Cell 18: Preview Label Array (Y)

Prints the `Y` array containing all sentiment labels.

- Values are either `0` (Negative) or `1` (Positive).
- Shape: `(1600000,)` — matches the number of samples in X.

In [24]:
print(Y)

[0 0 0 ... 1 1 1]


Splitting the data into training data and test Data

## Cell 19: Train-Test Split

Splits the dataset into **training** and **testing** sets:

| Parameter | Value | Meaning |
|---|---|---|
| `test_size=0.2` | 20% | 320,000 tweets reserved for testing |
| `random_state=2` | Fixed seed | Ensures same split every run (reproducibility) |

- **Training set (X_train, y_train):** 1,280,000 tweets — used to train the model
- **Test set (X_test, y_test):** 320,000 tweets — used to evaluate on unseen data

In [25]:
X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=2)

## Cell 20: Verify Split Shapes

Confirms the array sizes after splitting:

- `X.shape` = `(1600000,)` — Full dataset
- `X_train.shape` = `(1280000,)` — 80% for training
- `X_test.shape` = `(320000,)` — 20% for testing

In [26]:
print(X.shape,X_train.shape,X_test.shape)

(1600000,) (1280000,) (320000,)


## Cell 21: Preview Training Data

Prints a sample of the training set to verify the content looks correct — clean stemmed text strings ready for vectorization.

In [27]:
print(X_train)

<StringArray>
[                                                                                                     'bad feel',
                                                                                                  'love weekend',
                                                                  'littlew bit far day trip fun get muddi today',
                                                                        'jord passport inclin would ala neither',
                                                                                    'yyusuff open pic tri later',
                                                           'day prom think im gonna eat top ramen watch tv movi',
                                                                                            'follow pevert fail',
                                   'everyon elevensestim sorri well christinefarm hope better soon love sunshin',
                                                      'wilt air con still 

## Cell 22: Preview Test Data

Prints a sample of the test set. This data is **never seen** by the model during training — used only for final evaluation to measure real-world performance.

In [28]:
print(X_test)

<StringArray>
['brodiejay oh im go wow mona vale real place afteral know suck mvill slow train pffft',
                                                                            'babi grow',
                                                          'paint black roll stone best',
                                                                         'kk log byezz',
                                                      'shitti shitti shitti news today',
                                                                  'askmewhat hug wrong',
                                                      'samverril ipod touch care great',
                                                       'haha meant jog job ahh sunburn',
                                               'heard upset spoiler sytycd top depress',
 'must evalu black outfit hot replac lighter colour golf shirt still pop collar though',
 ...
            'listen bol whilst chat technog hk window live messeng gonna get coco mmmm',
  

## Cell 23: TF-IDF Vectorization

Converts text data into **numerical feature vectors** using TF-IDF (Term Frequency–Inverse Document Frequency):

```python
vectorizer = TfidfVectorizer()        # Create vectorizer
vectorizer.fit(X_train)               # Learn vocabulary from training data ONLY
X_train = vectorizer.transform(X_train)  # Transform training set
X_test = vectorizer.transform(X_test)    # Transform test set with same vocabulary
```

**Why TF-IDF?**
- Machine learning models cannot process raw text — they require numbers.
- TF-IDF assigns higher weights to words frequent in a tweet but rare overall (more informative).
- `.fit()` is called **only on training data** to prevent **data leakage**.

**Output:** Sparse matrix of shape `(1280000, ~461018)` — 1.28M tweets × ~461K unique word features.

In [29]:
#converting the textual data to numerical data
vectorizer=TfidfVectorizer()
vectorizer.fit(X_train)
X_train=vectorizer.transform(X_train)
X_test=vectorizer.transform(X_test)

## Cell 24: Inspect TF-IDF Matrix

Prints the TF-IDF sparse matrix structure.

- **Sparse matrix** stores only non-zero values to save memory (most tweets don't use most vocabulary words).
- Format: `(row_index, column_index) → tfidf_weight`
- The high number of stored elements (~9.4 million) reflects the vocabulary coverage across all training tweets.

In [30]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9452454 stored elements and shape (1280000, 461018)>
  Coords	Values
  (0, 31233)	0.7519171067329923
  (0, 129956)	0.6592576617698773
  (1, 241525)	0.5942540519072339
  (1, 437910)	0.8042773910733999
  (2, 42838)	0.2294043092907454
  (2, 93639)	0.16612263205577085
  (2, 128288)	0.2817039475788181
  (2, 140234)	0.2237237868613854
  (2, 145727)	0.16444046079173227
  (2, 237072)	0.6367700241935805
  (2, 278606)	0.48531397707654644
  (2, 411086)	0.18464287904169108
  (2, 416219)	0.2970321685021027
  (3, 8560)	0.37668420201866026
  (3, 179097)	0.4788299006944713
  (3, 201928)	0.524207488360723
  (3, 286683)	0.3682809811482607
  (3, 308910)	0.41292329558269175
  (3, 445747)	0.2188627839232472
  (4, 227660)	0.3291530007938817
  (4, 302738)	0.35966531233864635
  (4, 315347)	0.3291134109458034
  (4, 415818)	0.27525076285252903
  (4, 457632)	0.7604081439946883
  (5, 93639)	0.181548748438646
  :	:
  (1279996, 443184)	0.2023842381266662

Traning the ML MODEL

## Cell 25: Initialize Logistic Regression Model

Creates a **Logistic Regression** classifier:

- `max_iter=1000` — Allows up to 1000 iterations for convergence (default 100 is insufficient for large datasets).

**Why Logistic Regression for text classification?**
- Fast and memory-efficient on sparse TF-IDF matrices
- Outputs probability scores (used for confidence display)
- Works well for high-dimensional feature spaces
- Easy to interpret — word weights show which words drive sentiment

In [31]:
model=LogisticRegression(max_iter=1000)

## Cell 26: Train the Model

Trains the Logistic Regression model on the training data.

- **Input X_train:** TF-IDF feature matrix (shape: 1,280,000 × 461,018)
- **Input y_train:** Sentiment labels (0 or 1)
- The model learns which words are most associated with positive vs. negative sentiment.
- Training on 1.28 million examples may take a few minutes.

In [32]:
model.fit(X_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

## Cell 27: Training Accuracy

Evaluates the model's performance on the **training data**.

- **Output:** ~80.24% accuracy
- The model correctly predicts sentiment for approximately 80% of the tweets it was trained on.
- This is the **in-sample** accuracy — expected to be higher than test accuracy.

In [33]:
# Accuracy on training data
X_train_pred = model.predict(X_train)
training_data_accuracy = accuracy_score(y_train, X_train_pred)
print('Accuracy score on the training data :', training_data_accuracy)

Accuracy score on the training data : 0.802390625


## Cell 28: Saving the Model and Vectorizer

Saves both the trained model and the TF-IDF vectorizer to disk using **pickle**:

| File | Contents |
|---|---|
| `trained_model.sav` | Logistic Regression model weights |
| `vectorizer.sav` | Fitted TF-IDF vocabulary and IDF scores |

**Why save both?**
Any new tweet must be transformed using the **same vocabulary** learned during training. Without saving the vectorizer, feature dimensions would not match the model's expectations.

In [34]:
# Saving the trained model
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))

# Saving the vectorizer
pickle.dump(vectorizer, open('vectorizer.sav', 'wb'))

## Cell 29: Test Accuracy

Evaluates the model on **unseen test data** — the true measure of generalization.

- **Output:** ~77.66% accuracy
- The model correctly predicts sentiment for ~77.7% of tweets it has never seen before.
- The small gap between training (80.2%) and test (77.7%) accuracy indicates the model generalizes well without significant overfitting.

In [35]:
# Accuracy on test data
X_test_pred = model.predict(X_test)
test_data_accuracy = accuracy_score(y_test, X_test_pred)
print('Accuracy score on the test data :', test_data_accuracy)

Accuracy score on the test data : 0.77656875
